# AquaPINN Colab — Sequim Quickstart

Run the supplied time-corrected Sequim detections through MAT conversion and tracking. AML is the default for a short first run; NN and NN+ are available in the parameter block.

This follows the desktop Simple Example output layout: results, track CSV and pickle sidecar, figures, effective parameters, logs, and a run.json that the desktop Simple Example can reopen.

The example uses the fixed desktop demo tag G726777EF at TagID 2 and a horizontal fit. Confirm permission to process the supplied data in a hosted Colab runtime.

## 1. Source, runtime, and persistent output

Set REPO_URL in the next cell after the source repository has a public GitHub URL. Colab writes to Google Drive by default; local Jupyter writes to a local run folder.

In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, shutil, subprocess, sys, uuid

REPO_URL = ""  # Set this to the public GitHub clone URL before publishing.
REPO_REF = "main"

def find_repo():
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "AquaPINN_Tracker3D").is_dir() and (p / "Examples").is_dir():
            return p
    return None

REPO_DIR = find_repo()
if REPO_DIR is None:
    if not REPO_URL:
        raise RuntimeError("Set REPO_URL to the public repository URL, then rerun this cell.")
    REPO_DIR = Path("/content/aquapinn-source")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF,
                        REPO_URL, str(REPO_DIR)], check=True)
    if not (REPO_DIR / "AquaPINN_Tracker3D").is_dir():
        raise RuntimeError("Repository is missing AquaPINN_Tracker3D/")

try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount("/content/drive")
    OUTPUT_PARENT = Path("/content/drive/MyDrive/AquaPINN_Results")
except ImportError:
    IN_COLAB = False
    OUTPUT_PARENT = Path.cwd() / "AquaPINN_Colab_Runs"
OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)

def new_run(label):
    run_id = uuid.uuid4().hex
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    path = OUTPUT_PARENT / f"{label}_{stamp}_{run_id[:8]}"
    path.mkdir(parents=True, exist_ok=False)
    return path, run_id

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def write_json(path, obj):
    Path(path).write_text(json.dumps(obj, indent=2, default=str) + "\n", encoding="utf-8")

print("Project:", REPO_DIR)
print("Persistent output parent:", OUTPUT_PARENT)


## 2. Install dependencies

In [ ]:
%pip -q install "numpy<2.5" scipy pandas matplotlib utm networkx scikit-learn tqdm six tomli-w

In [ ]:
import torch
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if sys.version_info < (3, 11):
    raise RuntimeError("Python 3.11 or later is recommended.")

## 3. Study setup and main parameters

Edit the values in this block before running. For custom inputs stored in Drive, replace the bundled paths with their full Drive paths. TagID 2 is kept fixed so the resulting run record remains compatible with the desktop Simple Example.

In [ ]:
import tomllib
import tomli_w
import pandas as pd
from pathlib import Path

RUN_DIR, RUN_ID = new_run("Sequim")
SEQ_RUN = REPO_DIR / "Examples/Sequim/run"
PARAMS_FILE = SEQ_RUN / "params.toml"
with PARAMS_FILE.open("rb") as stream:
    PARAMS = tomllib.load(stream)

def from_example(key):
    value = PARAMS[key][0]
    return str((SEQ_RUN / value).resolve()) if value else ""

SETUP = {
    "mat_file": str((REPO_DIR / "Examples/Sequim/data/DriftingTags_timeCorrected.mat").resolve()),
    "phone_file": from_example("phoneFile"),
    "tag_file": from_example("tagFile"),
    "test_file": from_example("testFile"),
    "gps_file": from_example("GPSFile"),
    "tag_id": 2,
    "method": "AML",                 # AML, NN, or NNplus
    "nnplus_samples": 5,             # used only for NNplus
    "time_start": PARAMS["time_start"][0],
    "time_end": PARAMS["time_end"][0],
    "time_zone_hours": int(PARAMS["timeZone"][0]),
    "temperature_c": float(PARAMS["temperature_ref"][0]),
    "horizontal_only": True,
    "device": "Automatic",           # Automatic or CPU only
}
display(pd.DataFrame(SETUP.items(), columns=["Parameter", "Value"]))

Check the paths, tag, time window, temperature, dimension, method, and device. NN+ uses repeated fits; its spread is sample variability, not a calibrated confidence interval.

In [ ]:
from pathlib import Path
import tomli_w
import pandas as pd

MAT_FILE = Path(SETUP["mat_file"]).expanduser().resolve()
INPUTS = RUN_DIR / "inputs"
INPUTS.mkdir(parents=True, exist_ok=True)
MAT_STAGED = INPUTS / MAT_FILE.name
shutil.copy2(MAT_FILE, MAT_STAGED)
PHONE_FILE = Path(SETUP["phone_file"]).expanduser().resolve()
TAG_FILE = Path(SETUP["tag_file"]).expanduser().resolve()
TEST_FILE = Path(SETUP["test_file"]).expanduser().resolve() if SETUP["test_file"] else None
GPS_FILE = Path(SETUP["gps_file"]).expanduser().resolve() if SETUP["gps_file"] else None
for label, path in [("MAT detections", MAT_FILE), ("phone file", PHONE_FILE), ("tag file", TAG_FILE)]:
    if not path.is_file():
        raise FileNotFoundError(f"{label} not found: {path}")
if TEST_FILE and not TEST_FILE.is_file():
    raise FileNotFoundError(TEST_FILE)
if GPS_FILE and not GPS_FILE.is_file():
    raise FileNotFoundError(GPS_FILE)
catalogue = pd.read_csv(TAG_FILE)
if len(catalogue) <= 2 or str(catalogue.iloc[2]["Code"]).strip() != "G726777EF":
    raise ValueError("Desktop-compatible Sequim preset expects G726777EF at TagID 2.")

METHOD = SETUP["method"]
ENGINE_METHOD = "AML" if METHOD == "AML" else "NN"
SAMPLES = 1 if METHOD != "NNplus" else int(SETUP["nnplus_samples"])
RESULTS = RUN_DIR / "results"
PARAMS.update({
    "TagID": [2],
    "phoneFile": [str(PHONE_FILE)],
    "tagFile": [str(TAG_FILE)],
    "testFile": [str(TEST_FILE) if TEST_FILE else ""],
    "GPSFile": [str(GPS_FILE) if GPS_FILE else ""],
    "dataFolder": [str(INPUTS)],
    "output_folder": [str(RESULTS)],
    "time_start": [SETUP["time_start"]],
    "time_end": [SETUP["time_end"]],
    "timeZone": [int(SETUP["time_zone_hours"])],
    "temperature_ref": [float(SETUP["temperature_c"])],
    "ignoreZ": ["True" if SETUP["horizontal_only"] else "False"],
    "trackMethod": [ENGINE_METHOD],
    "nSamples": [SAMPLES],
})
EFFECTIVE_PARAMS = RUN_DIR / "params_effective.toml"
EFFECTIVE_PARAMS.write_text(tomli_w.dumps(PARAMS), encoding="utf-8")
write_json(RUN_DIR / "params_effective.json", PARAMS)
TRACK_REL = f"results/Track/track/track_Tag2_G726777EF_{METHOD}.csv"
RUN_RECORD = {"schema": 1, "run_id": RUN_ID, "state": "starting",
              "method": METHOD, "track_csv": TRACK_REL}
write_json(RUN_DIR / "run.json", RUN_RECORD)
write_json(RUN_DIR / "source-input-hashes.json",
           {str(p): sha256(p) for p in [MAT_FILE, MAT_STAGED, PHONE_FILE, TAG_FILE] +
            [p for p in [TEST_FILE, GPS_FILE] if p]})
print("Run folder:", RUN_DIR)
print("Tag G726777EF (TagID 2),", METHOD, "| fits:", SAMPLES)
print("Results:", RESULTS)

## 4. Run conversion and tracking

Sequim starts from corrected detection times, so this workflow converts the MAT input and tracks without running a new synchronization stage.

In [ ]:
import subprocess

write_json(RUN_DIR / "worker.json", {"run_id": RUN_ID, "state": "running",
                                      "stage": "Converting detections and tracking"})
env = os.environ.copy()
if SETUP["device"] == "CPU only":
    env["CUDA_VISIBLE_DEVICES"] = ""
launcher_source = (SEQ_RUN / "main_Sequim.py").read_text(encoding="utf-8")
launcher_source = launcher_source.replace(
    'SCRIPT_DIR   = Path(__file__).resolve().parent',
    f'SCRIPT_DIR   = Path(r"{SEQ_RUN}")')
launcher_source = launcher_source.replace(
    'param_file = str(SCRIPT_DIR / "params.toml")',
    f'param_file = r"{RUN_DIR / "params_effective.toml"}"')
launcher = RUN_DIR / "colab_launcher.py"
launcher.write_text(launcher_source, encoding="utf-8")
env["PYTHONPATH"] = str(SEQ_RUN) + os.pathsep + env.get("PYTHONPATH", "")
with (RUN_DIR / "console.log").open("w", encoding="utf-8") as log:
    completed = subprocess.run([sys.executable, str(launcher)], cwd=SEQ_RUN,
                               env=env, stdout=log, stderr=subprocess.STDOUT, text=True)
track_csv = RUN_DIR / TRACK_REL
if completed.returncode != 0 or not track_csv.is_file() or not Path(str(track_csv) + ".pickle").is_file():
    RUN_RECORD["state"] = "failed"
    write_json(RUN_DIR / "run.json", RUN_RECORD)
    write_json(RUN_DIR / "worker.json", {"run_id": RUN_ID, "state": "failed",
                                          "exit_code": completed.returncode})
    raise RuntimeError(f"Pipeline failed; inspect {RUN_DIR / 'console.log'}")
RUN_RECORD.update(state="completed", track_sha256=sha256(track_csv))
write_json(RUN_DIR / "run.json", RUN_RECORD)
write_json(RUN_DIR / "worker.json", {"run_id": RUN_ID, "state": "completed",
                                      "track_csv": TRACK_REL,
                                      "track_sha256": RUN_RECORD["track_sha256"]})
print("Completed:", track_csv)
print("Open in desktop Simple Example with:", RUN_DIR / "run.json")

## 5. Review and save

The run folder is already saved under the selected output parent. It contains the track CSV and pickle sidecar, figures, effective parameters, input hashes, engine log, console log, and desktop Simple Example run record.

In [ ]:
from IPython.display import display, Image
track = pd.read_csv(track_csv)
display(track.head())
plots = sorted(RESULTS.rglob("*.jpg"))
print("Figures:", len(plots))
for path in plots[:4]:
    display(Image(filename=str(path)))
print("Run folder:", RUN_DIR)